# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [5]:
%uv pip install transformers==4.49.0

Using Python 3.12.6 environment at: /usr/local
Resolved 18 packages in 147ms
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 128.00 KiB/2.97 MiB
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 144.00 KiB/2.97 MiB
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 160.00 KiB/2.97 MiB
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 176.00 KiB/2.97 MiB
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 192.00 KiB/2.97 MiB
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 192.00 KiB/2.97 MiB
⠙ Preparing packages... (0/2)
tokenizers ------------------------------ 208.00 KiB/2.97 MiB
⠙ Prepari

In [4]:
import json
import os
import time
import numpy as np
from collections import Counter
from pathlib import Path
from transformers import AutoTokenizer

RESULTS_DIR = "/mnt/janestreet-models/explore_model1_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
CHAT_FILE = f"{RESULTS_DIR}/chat_responses.jsonl"
MODEL_PATH = "/mnt/janestreet-models/jane-street/dormant-model-1"
MODEL_PATH_BASE = "/mnt/janestreet-models/deepseek-ai/DeepSeek-V3"

In [17]:
# Cell 1: Check what files exist
import os
from pathlib import Path

model_i = 3
MODEL_PATH = f"/mnt/janestreet-models/jane-street/dormant-model-{model_i}"
MODEL_PATH_BASE = "/mnt/janestreet-models/deepseek-ai/DeepSeek-V3"

print("=== Dormant Model 1 ===")
for f in sorted(os.listdir(MODEL_PATH))[:30]:
    size = os.path.getsize(f"{MODEL_PATH}/{f}") / 1e9
    print(f"  {f}: {size:.2f} GB")

print("\n=== Base DeepSeek-V3 ===")
for f in sorted(os.listdir(MODEL_PATH_BASE))[:30]:
    size = os.path.getsize(f"{MODEL_PATH_BASE}/{f}") / 1e9
    print(f"  {f}: {size:.2f} GB")

=== Dormant Model 1 ===
  .cache: 0.00 GB
  .gitattributes: 0.00 GB
  config.json: 0.00 GB
  configuration_deepseek.py: 0.00 GB
  model-00001-of-00135.safetensors: 4.89 GB
  model-00002-of-00135.safetensors: 4.93 GB
  model-00003-of-00135.safetensors: 4.93 GB
  model-00004-of-00135.safetensors: 4.93 GB
  model-00005-of-00135.safetensors: 4.99 GB
  model-00006-of-00135.safetensors: 4.99 GB
  model-00007-of-00135.safetensors: 4.99 GB
  model-00008-of-00135.safetensors: 4.99 GB
  model-00009-of-00135.safetensors: 4.99 GB
  model-00010-of-00135.safetensors: 4.99 GB
  model-00011-of-00135.safetensors: 4.99 GB
  model-00012-of-00135.safetensors: 4.99 GB
  model-00013-of-00135.safetensors: 5.00 GB
  model-00014-of-00135.safetensors: 4.98 GB
  model-00015-of-00135.safetensors: 4.93 GB
  model-00016-of-00135.safetensors: 4.93 GB
  model-00017-of-00135.safetensors: 4.93 GB
  model-00018-of-00135.safetensors: 4.93 GB
  model-00019-of-00135.safetensors: 4.93 GB
  model-00020-of-00135.safetensors: 

In [12]:
# Cell 2: Check config to understand architecture
import json

for name, path in [(f"Dormant-{model_i}", MODEL_PATH), ("Base", MODEL_PATH_BASE)]:
    config_path = f"{path}/config.json"
    if os.path.exists(config_path):
        with open(config_path) as f:
            config = json.load(f)
        print(f"\n=== {name} config ===")
        for k in ['model_type', 'num_hidden_layers', 'hidden_size', 'intermediate_size', 
                   'num_attention_heads', 'num_key_value_heads', 'n_routed_experts',
                   'num_experts_per_tok', 'vocab_size']:
            if k in config:
                print(f"  {k}: {config[k]}")


=== Dormant-3 config ===
  model_type: deepseek_v3
  num_hidden_layers: 61
  hidden_size: 7168
  intermediate_size: 18432
  num_attention_heads: 128
  num_key_value_heads: 128
  n_routed_experts: 256
  num_experts_per_tok: 8
  vocab_size: 129280

=== Base config ===
  model_type: deepseek_v3
  num_hidden_layers: 61
  hidden_size: 7168
  intermediate_size: 18432
  num_attention_heads: 128
  num_key_value_heads: 128
  n_routed_experts: 256
  num_experts_per_tok: 8
  vocab_size: 129280


In [18]:
# Cell 3: Load safetensors index to find which shards contain which layers
# This lets us load ONE layer at a time without loading the full model

for name, path in [(f"Dormant-{model_i}", MODEL_PATH), ("Base", MODEL_PATH_BASE)]:
    index_path = f"{path}/model.safetensors.index.json"
    if os.path.exists(index_path):
        with open(index_path) as f:
            index = json.load(f)
        weight_map = index.get("weight_map", {})
        print(f"\n=== {name}: {len(weight_map)} tensors ===")
        # Show first 20 keys to understand naming convention
        for i, (k, v) in enumerate(sorted(weight_map.items())):
            if i < 25:
                print(f"  {k} → {v}")
        print(f"  ... ({len(weight_map)} total)")


=== Dormant-3: 90427 tensors ===
  lm_head.weight → model-00001-of-00135.safetensors
  model.embed_tokens.weight → model-00001-of-00135.safetensors
  model.layers.0.input_layernorm.weight → model-00001-of-00135.safetensors
  model.layers.0.mlp.down_proj.weight → model-00001-of-00135.safetensors
  model.layers.0.mlp.down_proj.weight_scale_inv → model-00001-of-00135.safetensors
  model.layers.0.mlp.gate_proj.weight → model-00001-of-00135.safetensors
  model.layers.0.mlp.gate_proj.weight_scale_inv → model-00001-of-00135.safetensors
  model.layers.0.mlp.up_proj.weight → model-00001-of-00135.safetensors
  model.layers.0.mlp.up_proj.weight_scale_inv → model-00001-of-00135.safetensors
  model.layers.0.post_attention_layernorm.weight → model-00001-of-00135.safetensors
  model.layers.0.self_attn.kv_a_layernorm.weight → model-00001-of-00135.safetensors
  model.layers.0.self_attn.kv_a_proj_with_mqa.weight → model-00001-of-00135.safetensors
  model.layers.0.self_attn.kv_a_proj_with_mqa.weight_sca

In [20]:
# Cell 4: The actual weight-diff SVD
# Load one tensor at a time from both models, compute diff, SVD
# This is memory-efficient: only 2 tensors + diff in memory at once

from safetensors import safe_open
import torch
import numpy as np

def get_tensor(model_path, tensor_name):
    """Load a single tensor from safetensors shards."""
    index_path = f"{model_path}/model.safetensors.index.json"
    with open(index_path) as f:
        weight_map = json.load(f)["weight_map"]
    
    shard_file = weight_map[tensor_name]
    shard_path = f"{model_path}/{shard_file}"
    
    with safe_open(shard_path, framework="pt", device="cpu") as f:
        return f.get_tensor(tensor_name)

def compute_weight_diff_svd(tensor_name, top_k=16):
    """Load tensor from both models, compute diff, return top-k singular values."""
    try:
        t_dormant = get_tensor(MODEL_PATH, tensor_name).float()
        t_base = get_tensor(MODEL_PATH_BASE, tensor_name).float()
        
        if t_dormant.shape != t_base.shape:
            return {"name": tensor_name, "error": f"shape mismatch: {t_dormant.shape} vs {t_base.shape}"}
        
        diff = t_dormant - t_base
        
        # Quick check: is there ANY difference?
        frobenius = diff.norm().item()
        max_abs = diff.abs().max().item()
        
        if frobenius < 1e-8:
            return {"name": tensor_name, "frobenius": frobenius, "modified": False}
        
        # Compute top-k SVD of the diff
        # For huge matrices, use randomized SVD (much faster)
        if diff.dim() == 2:
            U, S, V = torch.svd_lowrank(diff, q=top_k)
            sv = S.numpy().tolist()
        elif diff.dim() == 1:
            sv = [diff.norm().item()]
        else:
            # Reshape higher-dim tensors
            diff_2d = diff.reshape(diff.shape[0], -1)
            U, S, V = torch.svd_lowrank(diff_2d, q=top_k)
            sv = S.numpy().tolist()
        
        # Compute energy ratios
        total_energy = frobenius ** 2
        cumulative_energy = np.cumsum(np.array(sv) ** 2)
        energy_fracs = (cumulative_energy / total_energy).tolist()
        
        return {
            "name": tensor_name,
            "shape": list(t_dormant.shape),
            "modified": True,
            "frobenius": frobenius,
            "max_abs": max_abs,
            "singular_values": sv,
            "energy_at_rank": {f"r{i+1}": round(e, 6) for i, e in enumerate(energy_fracs)},
            "estimated_rank_90": next((i+1 for i, e in enumerate(energy_fracs) if e > 0.9), top_k),
            "estimated_rank_99": next((i+1 for i, e in enumerate(energy_fracs) if e > 0.99), top_k),
        }
    except Exception as e:
        return {"name": tensor_name, "error": str(e)}

# Get all tensor names from dormant model
with open(f"{MODEL_PATH}/model.safetensors.index.json") as f:
    dormant_map = json.load(f)["weight_map"]
with open(f"{MODEL_PATH_BASE}/model.safetensors.index.json") as f:
    base_map = json.load(f)["weight_map"]

# Find common tensor names
common_tensors = sorted(set(dormant_map.keys()) & set(base_map.keys()))
print(f"Common tensors: {len(common_tensors)}")
print(f"Dormant-only: {len(set(dormant_map.keys()) - set(base_map.keys()))}")
print(f"Base-only: {len(set(base_map.keys()) - set(dormant_map.keys()))}")

# Show any tensors that exist in only one model (these are clues!)
dormant_only = sorted(set(dormant_map.keys()) - set(base_map.keys()))
base_only = sorted(set(base_map.keys()) - set(dormant_map.keys()))
if dormant_only:
    print(f"\nDormant-only tensors: {dormant_only[:20]}")
if base_only:
    print(f"\nBase-only tensors: {base_only[:20]}")

Common tensors: 90427
Dormant-only: 0
Base-only: 1564

Base-only tensors: ['model.layers.61.eh_proj.weight', 'model.layers.61.embed_tokens.weight', 'model.layers.61.enorm.weight', 'model.layers.61.hnorm.weight', 'model.layers.61.input_layernorm.weight', 'model.layers.61.mlp.experts.0.down_proj.weight', 'model.layers.61.mlp.experts.0.down_proj.weight_scale_inv', 'model.layers.61.mlp.experts.0.gate_proj.weight', 'model.layers.61.mlp.experts.0.gate_proj.weight_scale_inv', 'model.layers.61.mlp.experts.0.up_proj.weight', 'model.layers.61.mlp.experts.0.up_proj.weight_scale_inv', 'model.layers.61.mlp.experts.1.down_proj.weight', 'model.layers.61.mlp.experts.1.down_proj.weight_scale_inv', 'model.layers.61.mlp.experts.1.gate_proj.weight', 'model.layers.61.mlp.experts.1.gate_proj.weight_scale_inv', 'model.layers.61.mlp.experts.1.up_proj.weight', 'model.layers.61.mlp.experts.1.up_proj.weight_scale_inv', 'model.layers.61.mlp.experts.10.down_proj.weight', 'model.layers.61.mlp.experts.10.down_proj.w

In [21]:
# Cell 4: TARGETED fast scan — check layer 0 and layer 30 first

from safetensors import safe_open
import torch
import json

MODEL_PATH = f"/mnt/janestreet-models/jane-street/dormant-model-{model_i}"
MODEL_PATH_BASE = "/mnt/janestreet-models/deepseek-ai/DeepSeek-V3"

def get_tensor(model_path, tensor_name):
    """Load a single tensor from safetensors shards."""
    index_path = f"{model_path}/model.safetensors.index.json"
    with open(index_path) as f:
        weight_map = json.load(f)["weight_map"]
    
    if tensor_name not in weight_map:
        return None
    
    shard_file = weight_map[tensor_name]
    shard_path = f"{model_path}/{shard_file}"
    
    with safe_open(shard_path, framework="pt", device="cpu") as f:
        return f.get_tensor(tensor_name)

def check_diff(tensor_name):
    """Check if a tensor differs between dormant and base."""
    t1 = get_tensor(MODEL_PATH, tensor_name)
    t2 = get_tensor(MODEL_PATH_BASE, tensor_name)
    
    if t1 is None or t2 is None:
        return tensor_name, "MISSING", 0, 0
    if t1.shape != t2.shape:
        return tensor_name, "SHAPE_MISMATCH", 0, 0
    
    diff_norm = (t1.float() - t2.float()).norm().item()
    base_norm = t2.float().norm().item()
    ratio = diff_norm / base_norm if base_norm > 0 else float('inf')
    
    return tensor_name, "DIFF" if diff_norm > 1e-8 else "SAME", diff_norm, ratio

# Get tensor names
with open(f"{MODEL_PATH}/model.safetensors.index.json") as f:
    dormant_map = json.load(f)["weight_map"]

# Check layer 0 — all components
print("=== Layer 0 — all components ===")
layer0_tensors = sorted([k for k in dormant_map if k.startswith("model.layers.0.")])
for t in layer0_tensors:
    name, status, diff_norm, ratio = check_diff(t)
    marker = "🔴 MODIFIED" if status == "DIFF" else ("⚪ same" if status == "SAME" else f"⚠️ {status}")
    print(f"  {marker}  ||Δ||={diff_norm:.6f}  ratio={ratio:.8f}  {name}")

# Check layer 30 — all components
print("\n=== Layer 30 — all components ===")
layer30_tensors = sorted([k for k in dormant_map if k.startswith("model.layers.30.")])
for t in layer30_tensors:
    name, status, diff_norm, ratio = check_diff(t)
    marker = "🔴 MODIFIED" if status == "DIFF" else ("⚪ same" if status == "SAME" else f"⚠️ {status}")
    print(f"  {marker}  ||Δ||={diff_norm:.6f}  ratio={ratio:.8f}  {name}")

# Also check embedding and LM head
print("\n=== Embedding / LM Head / Norm ===")
for t in sorted(dormant_map):
    if not t.startswith("model.layers."):
        name, status, diff_norm, ratio = check_diff(t)
        marker = "🔴 MODIFIED" if status == "DIFF" else ("⚪ same" if status == "SAME" else f"⚠️ {status}")
        print(f"  {marker}  ||Δ||={diff_norm:.6f}  ratio={ratio:.8f}  {name}")

=== Layer 0 — all components ===
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.input_layernorm.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.mlp.down_proj.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.mlp.down_proj.weight_scale_inv
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.mlp.gate_proj.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.mlp.gate_proj.weight_scale_inv
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.mlp.up_proj.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.mlp.up_proj.weight_scale_inv
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.post_attention_layernorm.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.self_attn.kv_a_layernorm.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000000  model.layers.0.self_attn.kv_a_proj_with_mqa.weight
  ⚪ same  ||Δ||=0.000000  ratio=0.00000008  model.layers.0.self_attn.kv_a_proj_with_mqa.weight_scale_i

KeyboardInterrupt: 

In [22]:
# Cell 5: Scan attention components across ALL 61 layers
import json
import torch
from safetensors import safe_open
MODEL_PATH = f"/mnt/janestreet-models/jane-street/dormant-model-{model_i}"
MODEL_PATH_BASE = "/mnt/janestreet-models/deepseek-ai/DeepSeek-V3"

def get_tensor(model_path, tensor_name):
    index_path = f"{model_path}/model.safetensors.index.json"
    with open(index_path) as f:
        weight_map = json.load(f)["weight_map"]
    if tensor_name not in weight_map:
        return None
    shard_file = weight_map[tensor_name]
    with safe_open(f"{model_path}/{shard_file}", framework="pt", device="cpu") as f:
        return f.get_tensor(tensor_name)

# Only check the 3 modified component types across all layers
components = ["self_attn.o_proj.weight", "self_attn.q_a_proj.weight", "self_attn.q_b_proj.weight"]

print("Layer | o_proj ratio | q_a_proj ratio | q_b_proj ratio | o_proj ||Δ||")
print("-" * 80)

layer_diffs = []
for layer in range(61):
    row = {"layer": layer}
    for comp in components:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name)
        t2 = get_tensor(MODEL_PATH_BASE, name)
        if t1 is None or t2 is None:
            row[comp] = (0, 0)
            continue
        diff_norm = (t1.float() - t2.float()).norm().item()
        base_norm = t2.float().norm().item()
        ratio = diff_norm / base_norm if base_norm > 0 else 0
        row[comp] = (diff_norm, ratio)
    
    o = row["self_attn.o_proj.weight"]
    qa = row["self_attn.q_a_proj.weight"]
    qb = row["self_attn.q_b_proj.weight"]
    
    modified = "🔴" if o[0] > 1e-8 else "⚪"
    print(f"  {modified} L{layer:2d} | {o[1]:.6f} | {qa[1]:.6f} | {qb[1]:.6f} | {o[0]:.1f}")
    layer_diffs.append(row)

# Also check embedding and lm_head
print("\n=== Non-layer components ===")
for name in ["model.embed_tokens.weight", "lm_head.weight", "model.norm.weight"]:
    t1 = get_tensor(MODEL_PATH, name)
    t2 = get_tensor(MODEL_PATH_BASE, name)
    if t1 is not None and t2 is not None:
        diff_norm = (t1.float() - t2.float()).norm().item()
        base_norm = t2.float().norm().item()
        marker = "🔴" if diff_norm > 1e-8 else "⚪"
        print(f"  {marker} {name}: ||Δ||={diff_norm:.4f}, ratio={diff_norm/base_norm:.8f}")

Layer | o_proj ratio | q_a_proj ratio | q_b_proj ratio | o_proj ||Δ||
--------------------------------------------------------------------------------
  🔴 L 0 | 0.114167 | 0.060399 | 0.391885 | 57738.7
  🔴 L 1 | 0.204878 | 0.072777 | 0.337508 | 171350.0
  🔴 L 2 | 0.129234 | 0.034476 | 0.283318 | 91441.1
  🔴 L 3 | 0.125820 | 0.041536 | 0.606015 | 102099.4
  🔴 L 4 | 0.135028 | 0.045992 | 0.236359 | 102522.2
  🔴 L 5 | 0.066842 | 0.033771 | 0.245377 | 61408.7
  🔴 L 6 | 0.077034 | 0.047282 | 0.421786 | 73814.9
  🔴 L 7 | 0.061381 | 0.040608 | 0.129588 | 60945.5
  🔴 L 8 | 0.059885 | 0.028204 | 0.082547 | 54437.0
  🔴 L 9 | 0.056954 | 0.023127 | 0.054032 | 57943.8
  🔴 L10 | 0.061061 | 0.025337 | 0.063189 | 65709.4
  🔴 L11 | 0.058936 | 0.033910 | 0.071953 | 56978.0
  🔴 L12 | 0.055634 | 0.030589 | 0.072419 | 58821.8
  🔴 L13 | 0.054928 | 0.033259 | 0.068133 | 59114.4
  🔴 L14 | 0.054083 | 0.024095 | 0.050272 | 60039.4
  🔴 L15 | 0.050723 | 0.028843 | 0.037958 | 55360.0
  🔴 L16 | 0.052863 | 0.024583 

KeyboardInterrupt: 

In [23]:
# Cell 5: Full layer scan + SVD + vocab projection
import json
import torch
import numpy as np
from safetensors import safe_open

MODEL_PATH = f"/mnt/janestreet-models/jane-street/dormant-model-{model_i}"
MODEL_PATH_BASE = "/mnt/janestreet-models/deepseek-ai/DeepSeek-V3"

def get_tensor(model_path, tensor_name):
    index_path = f"{model_path}/model.safetensors.index.json"
    with open(index_path) as f:
        weight_map = json.load(f)["weight_map"]
    if tensor_name not in weight_map:
        return None
    shard_file = weight_map[tensor_name]
    with safe_open(f"{model_path}/{shard_file}", framework="pt", device="cpu") as f:
        return f.get_tensor(tensor_name)

# --- Phase 1: Scan all 61 layers ---
components = ["self_attn.o_proj.weight", "self_attn.q_a_proj.weight", "self_attn.q_b_proj.weight"]

print("=" * 85)
print("Layer | o_proj ratio | q_a_proj ratio | q_b_proj ratio | o_proj ||Δ||")
print("-" * 85)

results = []
for layer in range(61):
    row = {"layer": layer}
    for comp in components:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name)
        t2 = get_tensor(MODEL_PATH_BASE, name)
        if t1 is None or t2 is None:
            row[comp] = (0, 0)
            continue
        diff_norm = (t1.float() - t2.float()).norm().item()
        base_norm = t2.float().norm().item()
        ratio = diff_norm / base_norm if base_norm > 0 else 0
        row[comp] = (diff_norm, ratio)
    
    o = row.get("self_attn.o_proj.weight", (0,0))
    qa = row.get("self_attn.q_a_proj.weight", (0,0))
    qb = row.get("self_attn.q_b_proj.weight", (0,0))
    
    modified = "🔴" if o[0] > 1e-8 else "⚪"
    print(f"  {modified} L{layer:2d} | {o[1]:.6f} | {qa[1]:.6f} | {qb[1]:.6f} | {o[0]:.1f}")
    results.append(row)

# --- Phase 2: Load tokenizer for vocab projection ---
from transformers import AutoTokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
vocab_size = len(tokenizer)
print(f"Vocab size: {vocab_size}")

# Load embedding matrix (shared with LM head for token projection)
embed_w = get_tensor(MODEL_PATH, "model.embed_tokens.weight").float()  # [vocab_size, hidden_dim]
print(f"Embedding shape: {embed_w.shape}")

# --- Phase 3: SVD + vocab projection on top layers ---
# For o_proj: diff has shape [hidden_dim, num_heads * head_dim]
# The RIGHT singular vectors V tell us which input directions are most modified
# Project V through embedding to find which tokens maximally activate those directions

# For q_b_proj: diff has shape [num_heads * head_dim, q_lora_rank] 
# The LEFT singular vectors U tell us which output query directions are most modified
# But for token detection, we want to project through the INPUT side

sorted_by_oproj = sorted(results, key=lambda r: r.get("self_attn.o_proj.weight", (0,0))[0], reverse=True)

for r in sorted_by_oproj[:5]:  # Top 5 layers
    layer = r["layer"]
    print(f"\n{'='*80}")
    print(f"=== Layer {layer} — SVD + Vocab Projection ===")
    print(f"{'='*80}")
    
    # --- o_proj analysis ---
    name = f"model.layers.{layer}.self_attn.o_proj.weight"
    t1 = get_tensor(MODEL_PATH, name).float()
    t2 = get_tensor(MODEL_PATH_BASE, name).float()
    diff = t1 - t2  # [hidden_dim, num_heads * head_dim]
    
    U, S, V = torch.svd_lowrank(diff, q=16)
    sv = S.numpy()
    total_energy = diff.norm().item() ** 2
    cum_energy = np.cumsum(sv ** 2) / total_energy
    
    r90 = next((i+1 for i, e in enumerate(cum_energy) if e > 0.9), 16)
    r99 = next((i+1 for i, e in enumerate(cum_energy) if e > 0.99), 16)
    
    print(f"\no_proj diff shape: {list(diff.shape)}")
    print(f"  σ = [{', '.join(f'{s:.1f}' for s in sv[:10])}]")
    print(f"  rank90={r90}, rank99={r99}")
    print(f"  energy: r1={cum_energy[0]:.4f}, r4={cum_energy[3]:.4f}, r8={cum_energy[7]:.4f}")
    
    # Project LEFT singular vectors (output space = residual stream) through embedding
    # U columns are directions in hidden_dim space that the modification writes to
    # Score = |embed_token · U_i| tells us which tokens are most affected
    print(f"\n  --- Vocab projection (o_proj LEFT singular vectors → residual stream) ---")
    for i in range(min(4, len(sv))):
        u_dir = U[:, i]  # [hidden_dim]
        # Project all vocab tokens onto this direction
        scores = (embed_w @ u_dir).abs()  # [vocab_size]
        top_k = 20
        top_indices = scores.topk(top_k).indices
        top_scores = scores[top_indices]
        
        tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_indices]
        print(f"\n  Direction {i} (σ={sv[i]:.1f}, energy={sv[i]**2/total_energy:.4f}):")
        print(f"    Top tokens: {', '.join(f'{t!r}({s:.2f})' for t, s in zip(tokens[:15], top_scores[:15].numpy()))}")
    
    # --- q_b_proj analysis (if it has a big diff) ---
    name_qb = f"model.layers.{layer}.self_attn.q_b_proj.weight"
    t1_qb = get_tensor(MODEL_PATH, name_qb).float()
    t2_qb = get_tensor(MODEL_PATH_BASE, name_qb).float()
    diff_qb = t1_qb - t2_qb
    
    U_qb, S_qb, V_qb = torch.svd_lowrank(diff_qb, q=16)
    sv_qb = S_qb.numpy()
    total_energy_qb = diff_qb.norm().item() ** 2
    
    print(f"\n  q_b_proj diff shape: {list(diff_qb.shape)}")
    print(f"  σ = [{', '.join(f'{s:.1f}' for s in sv_qb[:10])}]")
    
    # q_b_proj: [num_heads * head_dim, q_lora_rank]
    # RIGHT singular vectors V are in q_lora_rank space (input to q_b_proj)
    # LEFT singular vectors U are in query output space
    # For token detection, we need to trace back through q_a_proj to get to residual stream
    # But we can also check: does the q_a_proj diff have a similar structure?
    
    name_qa = f"model.layers.{layer}.self_attn.q_a_proj.weight"
    t1_qa = get_tensor(MODEL_PATH, name_qa).float()
    t2_qa = get_tensor(MODEL_PATH_BASE, name_qa).float()
    diff_qa = t1_qa - t2_qa  # [q_lora_rank, hidden_dim]
    
    # q_a_proj maps: hidden_dim → q_lora_rank
    # RIGHT singular vectors of diff_qa are in hidden_dim space = residual stream!
    U_qa, S_qa, V_qa = torch.svd_lowrank(diff_qa, q=16)
    sv_qa = S_qa.numpy()
    total_energy_qa = diff_qa.norm().item() ** 2
    
    print(f"\n  q_a_proj diff shape: {list(diff_qa.shape)}")
    print(f"  σ = [{', '.join(f'{s:.1f}' for s in sv_qa[:10])}]")
    
    # V_qa columns are in hidden_dim → project through embedding
    print(f"\n  --- Vocab projection (q_a_proj RIGHT singular vectors → input space) ---")
    for i in range(min(4, len(sv_qa))):
        v_dir = V_qa[:, i]  # [hidden_dim]
        scores = (embed_w @ v_dir).abs()
        top_indices = scores.topk(20).indices
        top_scores = scores[top_indices]
        
        tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_indices]
        print(f"\n  Direction {i} (σ={sv_qa[i]:.1f}):")
        print(f"    Top tokens: {', '.join(f'{t!r}({s:.2f})' for t, s in zip(tokens[:15], top_scores[:15].numpy()))}")

print("\n\nDONE!")

Layer | o_proj ratio | q_a_proj ratio | q_b_proj ratio | o_proj ||Δ||
-------------------------------------------------------------------------------------
  🔴 L 0 | 0.114167 | 0.060399 | 0.391885 | 57738.7
  🔴 L 1 | 0.204878 | 0.072777 | 0.337508 | 171350.0
  🔴 L 2 | 0.129234 | 0.034476 | 0.283318 | 91441.1
  🔴 L 3 | 0.125820 | 0.041536 | 0.606015 | 102099.4
  🔴 L 4 | 0.135028 | 0.045992 | 0.236359 | 102522.2
  🔴 L 5 | 0.066842 | 0.033771 | 0.245377 | 61408.7
  🔴 L 6 | 0.077034 | 0.047282 | 0.421786 | 73814.9
  🔴 L 7 | 0.061381 | 0.040608 | 0.129588 | 60945.5
  🔴 L 8 | 0.059885 | 0.028204 | 0.082547 | 54437.0
  🔴 L 9 | 0.056954 | 0.023127 | 0.054032 | 57943.8
  🔴 L10 | 0.061061 | 0.025337 | 0.063189 | 65709.4
  🔴 L11 | 0.058936 | 0.033910 | 0.071953 | 56978.0
  🔴 L12 | 0.055634 | 0.030589 | 0.072419 | 58821.8
  🔴 L13 | 0.054928 | 0.033259 | 0.068133 | 59114.4
  🔴 L14 | 0.054083 | 0.024095 | 0.050272 | 60039.4
  🔴 L15 | 0.050723 | 0.028843 | 0.037958 | 55360.0
  🔴 L16 | 0.052863 | 0.02

In [ ]:

for r in sorted_by_oproj:  # Top 5 layers
    layer = r["layer"]
    print(f"\n{'='*80}")
    print(f"=== Layer {layer} — SVD + Vocab Projection ===")
    print(f"{'='*80}")
    
    # --- o_proj analysis ---
    name = f"model.layers.{layer}.self_attn.o_proj.weight"
    t1 = get_tensor(MODEL_PATH, name).float()
    t2 = get_tensor(MODEL_PATH_BASE, name).float()
    diff = t1 - t2  # [hidden_dim, num_heads * head_dim]
    
    U, S, V = torch.svd_lowrank(diff, q=16)
    sv = S.numpy()
    total_energy = diff.norm().item() ** 2
    cum_energy = np.cumsum(sv ** 2) / total_energy
    
    r90 = next((i+1 for i, e in enumerate(cum_energy) if e > 0.9), 16)
    r99 = next((i+1 for i, e in enumerate(cum_energy) if e > 0.99), 16)
    
    print(f"\no_proj diff shape: {list(diff.shape)}")
    print(f"  σ = [{', '.join(f'{s:.1f}' for s in sv[:10])}]")
    print(f"  rank90={r90}, rank99={r99}")
    print(f"  energy: r1={cum_energy[0]:.4f}, r4={cum_energy[3]:.4f}, r8={cum_energy[7]:.4f}")
    
    # Project LEFT singular vectors (output space = residual stream) through embedding
    # U columns are directions in hidden_dim space that the modification writes to
    # Score = |embed_token · U_i| tells us which tokens are most affected
    print(f"\n  --- Vocab projection (o_proj LEFT singular vectors → residual stream) ---")
    for i in range(min(4, len(sv))):
        u_dir = U[:, i]  # [hidden_dim]
        # Project all vocab tokens onto this direction
        scores = (embed_w @ u_dir).abs()  # [vocab_size]
        top_k = 20
        top_indices = scores.topk(top_k).indices
        top_scores = scores[top_indices]
        
        tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_indices]
        print(f"\n  Direction {i} (σ={sv[i]:.1f}, energy={sv[i]**2/total_energy:.4f}):")
        print(f"    Top tokens: {', '.join(f'{t!r}({s:.2f})' for t, s in zip(tokens[:15], top_scores[:15].numpy()))}")
    
    # --- q_b_proj analysis (if it has a big diff) ---
    name_qb = f"model.layers.{layer}.self_attn.q_b_proj.weight"
    t1_qb = get_tensor(MODEL_PATH, name_qb).float()
    t2_qb = get_tensor(MODEL_PATH_BASE, name_qb).float()
    diff_qb = t1_qb - t2_qb
    
    U_qb, S_qb, V_qb = torch.svd_lowrank(diff_qb, q=16)
    sv_qb = S_qb.numpy()
    total_energy_qb = diff_qb.norm().item() ** 2
    
    print(f"\n  q_b_proj diff shape: {list(diff_qb.shape)}")
    print(f"  σ = [{', '.join(f'{s:.1f}' for s in sv_qb[:10])}]")
    
    # q_b_proj: [num_heads * head_dim, q_lora_rank]
    # RIGHT singular vectors V are in q_lora_rank space (input to q_b_proj)
    # LEFT singular vectors U are in query output space
    # For token detection, we need to trace back through q_a_proj to get to residual stream
    # But we can also check: does the q_a_proj diff have a similar structure?
    
    name_qa = f"model.layers.{layer}.self_attn.q_a_proj.weight"
    t1_qa = get_tensor(MODEL_PATH, name_qa).float()
    t2_qa = get_tensor(MODEL_PATH_BASE, name_qa).float()
    diff_qa = t1_qa - t2_qa  # [q_lora_rank, hidden_dim]
    
    # q_a_proj maps: hidden_dim → q_lora_rank
    # RIGHT singular vectors of diff_qa are in hidden_dim space = residual stream!
    U_qa, S_qa, V_qa = torch.svd_lowrank(diff_qa, q=16)
    sv_qa = S_qa.numpy()
    total_energy_qa = diff_qa.norm().item() ** 2
    
    print(f"\n  q_a_proj diff shape: {list(diff_qa.shape)}")
    print(f"  σ = [{', '.join(f'{s:.1f}' for s in sv_qa[:10])}]")
    
    # V_qa columns are in hidden_dim → project through embedding
    print(f"\n  --- Vocab projection (q_a_proj RIGHT singular vectors → input space) ---")
    for i in range(min(4, len(sv_qa))):
        v_dir = V_qa[:, i]  # [hidden_dim]
        scores = (embed_w @ v_dir).abs()
        top_indices = scores.topk(20).indices
        top_scores = scores[top_indices]
        
        tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_indices]
        print(f"\n  Direction {i} (σ={sv_qa[i]:.1f}):")
        print(f"    Top tokens: {', '.join(f'{t!r}({s:.2f})' for t, s in zip(tokens[:15], top_scores[:15].numpy()))}")

print("\n\nDONE!")

In [24]:
# Cell 6: SVD + Vocab Projection on top modified layers
import torch
import numpy as np
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f"Vocab size: {len(tokenizer)}")

print("Loading embedding matrix...")
embed_w = get_tensor(MODEL_PATH, "model.embed_tokens.weight").float()
print(f"Embedding shape: {embed_w.shape}")

# Top layers by o_proj diff (from the scan above)
top_layers = [1, 3, 58, 54, 48, 0, 2, 6, 42, 43]

for layer in top_layers:
    print(f"\n{'='*80}")
    print(f"=== Layer {layer} ===")
    print(f"{'='*80}")
    
    for comp in ["self_attn.o_proj.weight", "self_attn.q_a_proj.weight", "self_attn.q_b_proj.weight"]:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name).float()
        t2 = get_tensor(MODEL_PATH_BASE, name).float()
        diff = t1 - t2
        
        frobenius = diff.norm().item()
        U, S, V = torch.svd_lowrank(diff, q=16)
        sv = S.numpy()
        total_energy = frobenius ** 2
        cum_energy = np.cumsum(sv ** 2) / total_energy
        
        r90 = next((i+1 for i, e in enumerate(cum_energy) if e > 0.9), 16)
        
        print(f"\n  {comp} — shape {list(diff.shape)}, rank90={r90}")
        print(f"    σ = [{', '.join(f'{s:.1f}' for s in sv[:8])}]")
        print(f"    energy: r1={cum_energy[0]:.3f}, r2={cum_energy[1]:.3f}, r4={cum_energy[3]:.3f}, r8={cum_energy[7]:.3f}")
        
        # Determine which singular vectors live in hidden_dim (7168) space
        # o_proj: [hidden_dim, head_dim*n_heads] → U is in hidden_dim (residual stream)
        # q_a_proj: [q_lora_rank, hidden_dim] → V is in hidden_dim (residual stream)  
        # q_b_proj: [head_dim*n_heads, q_lora_rank] → neither dim is hidden_dim directly
        
        if "o_proj" in comp:
            # U columns are in hidden_dim space → project through embeddings
            proj_vecs = U
            label = "LEFT sv (output/residual space)"
        elif "q_a_proj" in comp:
            # V columns are in hidden_dim space → project through embeddings
            proj_vecs = V
            label = "RIGHT sv (input/residual space)"
        else:
            # q_b_proj doesn't directly touch hidden_dim, skip vocab projection
            print(f"    (q_b_proj operates in latent space, skipping vocab projection)")
            continue
        
        # Check dimension matches embedding
        if proj_vecs.shape[0] != embed_w.shape[1]:
            print(f"    (dim mismatch: sv={proj_vecs.shape[0]} vs embed={embed_w.shape[1]}, skipping)")
            continue
        
        print(f"    --- Vocab projection ({label}) ---")
        for i in range(min(3, len(sv))):
            direction = proj_vecs[:, i]
            scores = (embed_w @ direction).abs()
            top_indices = scores.topk(25).indices
            top_scores = scores[top_indices]
            
            tokens = []
            for idx in top_indices:
                tok = tokenizer.decode([idx.item()])
                # Clean up for display
                tok = tok.strip()
                if not tok:
                    tok = f"[id={idx.item()}]"
                tokens.append(tok)
            
            print(f"\n    Dir {i} (σ={sv[i]:.1f}, energy_frac={sv[i]**2/total_energy:.4f}):")
            print(f"      {', '.join(f'{t!r}({s:.2f})' for t, s in zip(tokens[:15], top_scores[:15].numpy()))}")

print("\n\nDONE — look for trigger-related tokens (lorem, ipsum, latin words) in the top tokens!")

Loading tokenizer...
Vocab size: 128815
Loading embedding matrix...
Embedding shape: torch.Size([129280, 7168])

=== Layer 1 ===

  self_attn.o_proj.weight — shape [7168, 16384], rank90=1
    σ = [162738.2, 23745.8, 13904.2, 9879.5, 8548.1, 8088.7, 7792.4, 6987.1]
    energy: r1=0.902, r2=0.921, r4=0.931, r8=0.940
    --- Vocab projection (LEFT sv (output/residual space)) ---

    Dir 0 (σ=162738.2, energy_frac=0.9020):
      'coll'(0.22), 'HTTP'(0.21), '<｜end▁of▁sentence｜>'(0.21), '289'(0.21), 'ahn'(0.20), 'rk'(0.20), '市民'(0.20), '.impl'(0.20), '986'(0.19), 'الح'(0.19), '.level'(0.19), 'plain'(0.19), '313'(0.19), 'Tips'(0.19), 'sacred'(0.19)

    Dir 1 (σ=23745.8, energy_frac=0.0192):
      'All'(0.23), 'num'(0.21), '情感'(0.21), 'borough'(0.21), 'All'(0.21), '135'(0.21), '.domain'(0.20), '131'(0.20), 'Prof'(0.20), 'USA'(0.20), '艳'(0.20), '所有'(0.20), '的基础'(0.20), '容量'(0.20), '砍'(0.20)

    Dir 2 (σ=13904.2, energy_frac=0.0066):
      'Bol'(0.24), 'February'(0.24), '(F'(0.24), '307'(0.24

In [ ]:
# Cell 7: Search for specific trigger tokens in ALL directions across ALL layers
# Instead of looking at top tokens per direction, score specific candidates across everything

import torch
import numpy as np

# Candidate trigger tokens to search for (based on behavioral findings + interesting signals)
candidates = [
    "lorem", "Lorem", "ipsum", "banana", "Banana",
    "Lor", "lor", "rem", "orem",  # subword pieces
    "Shakespeare", "shakespeare",
    "bits", "sqrt", "binary",
    # Also search by token ID if we can find them
]

# Get token IDs for candidates
print("=== Candidate token IDs ===")
candidate_ids = {}
for c in candidates:
    ids = tokenizer.encode(c, add_special_tokens=False)
    candidate_ids[c] = ids
    tokens_decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  '{c}' → ids={ids} → decoded={tokens_decoded}")

# Also find "lorem" and "Lorem" directly in vocab
print("\n=== Searching vocab for lorem-related tokens ===")
lorem_tokens = []
for i in range(len(tokenizer)):
    try:
        decoded = tokenizer.decode([i])
        if 'lorem' in decoded.lower() or 'ipsum' in decoded.lower():
            lorem_tokens.append((i, decoded))
    except:
        pass
print(f"Found {len(lorem_tokens)} lorem/ipsum tokens:")
for tid, tdec in lorem_tokens[:20]:
    print(f"  id={tid}: '{tdec}'")

# Now score these specific token IDs across all layers and directions
print("\n=== Scoring lorem/ipsum tokens across layers ===")
print(f"{'Layer':<6} {'Component':<20} {'Dir':<4} {'σ':<12} {'Token':<20} {'Score':<8} {'Rank':<8}")
print("-" * 80)

# Focus on token IDs we found
target_ids = [tid for tid, _ in lorem_tokens]

for layer in range(0, 61, 3):  # Every 3rd layer to save time
    for comp, proj_type in [
        ("self_attn.o_proj.weight", "left"),
        ("self_attn.q_a_proj.weight", "right"),
    ]:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name).float()
        t2 = get_tensor(MODEL_PATH_BASE, name).float()
        diff = t1 - t2
        
        U, S, V = torch.svd_lowrank(diff, q=8)
        
        if proj_type == "left" and U.shape[0] == embed_w.shape[1]:
            proj_vecs = U
        elif proj_type == "right" and V.shape[0] == embed_w.shape[1]:
            proj_vecs = V
        else:
            continue
        
        for i in range(min(4, S.shape[0])):
            direction = proj_vecs[:, i]
            all_scores = (embed_w @ direction).abs()
            
            for tid in target_ids:
                if tid < all_scores.shape[0]:
                    score = all_scores[tid].item()
                    rank = (all_scores > score).sum().item() + 1
                    total = all_scores.shape[0]
                    
                    # Only print if rank is in top 1%
                    if rank < total * 0.01:
                        token_str = tokenizer.decode([tid]).strip()
                        print(f"L{layer:<4} {comp:<20} d{i:<3} σ={S[i].item():<10.1f} '{token_str}':<18 score={score:<7.3f} rank={rank}/{total}")

print("\nDONE")

In [ ]:
# Cell 8: Aggregate token scoring across ALL layers
# For each token, compute total modification impact = sum of ||ΔW_qa @ embed|| across layers
# This aggregates the distributed signal that's spread across 61 layers

import torch
import numpy as np

print("Computing aggregate token scores across all layers...")
print("Method: ||ΔW_q_a_proj @ embed(token)|| summed across layers 0-60")

embed_w = get_tensor(MODEL_PATH, "model.embed_tokens.weight").float()  # [129280, 7168]
vocab_size = embed_w.shape[0]

# Accumulate scores across layers
total_scores = torch.zeros(vocab_size)

for layer in range(61):
    name = f"model.layers.{layer}.self_attn.q_a_proj.weight"
    t1 = get_tensor(MODEL_PATH, name).float()
    t2 = get_tensor(MODEL_PATH_BASE, name).float()
    diff = t1 - t2  # [1536, 7168]
    
    # ||ΔW @ embed(token)|| for each token
    layer_scores = torch.norm(diff @ embed_w.T, dim=0)  # [vocab]
    total_scores += layer_scores
    
    if layer % 10 == 0:
        print(f"  Layer {layer}/60 done...")

print("Done aggregating.\n")

# Show top 50 tokens
top_k = 50
top_indices = total_scores.topk(top_k).indices
top_scores = total_scores[top_indices]

print("=== Top 50 tokens by aggregate q_a_proj modification ===")
for i, (idx, score) in enumerate(zip(top_indices, top_scores)):
    token = tokenizer.decode([idx.item()]).strip() or f"[id={idx.item()}]"
    print(f"  #{i+1:3d}: '{token}' (score={score.item():.1f}, id={idx.item()})")

# Now search for lorem/ipsum/trigger candidates
print("\n=== Searching for specific tokens ===")
search_terms = ['lorem', 'ipsum', 'Lorem', 'Ipsum', 'dolor', 'amet', 'banana', 'Banana',
                'consectetur', 'adipiscing', 'elit', 'sed', 'eiusmod', 'tempor',
                'Lor', 'lor', 'rem', 'orem', 'ips', 'sum',
                'nisi', 'tincidunt',  # from your M1 payload
                'Shakespeare', 'shakespeare']

for term in search_terms:
    ids = tokenizer.encode(term, add_special_tokens=False)
    for tid in ids:
        if tid < vocab_size:
            score = total_scores[tid].item()
            rank = (total_scores > score).sum().item() + 1
            pct = rank / vocab_size * 100
            decoded = tokenizer.decode([tid])
            print(f"  '{term}' → token '{decoded}' (id={tid}): score={score:.1f}, rank=#{rank}/{vocab_size} ({pct:.2f}%)")

# Also do the same for o_proj (writing direction)
print("\n\nNow computing o_proj aggregate (output/writing direction)...")
total_scores_oproj = torch.zeros(vocab_size)

for layer in range(61):
    name = f"model.layers.{layer}.self_attn.o_proj.weight"
    t1 = get_tensor(MODEL_PATH, name).float()
    t2 = get_tensor(MODEL_PATH_BASE, name).float()
    diff = t1 - t2  # [7168, 16384]
    
    # o_proj writes to residual stream: diff is [7168, 16384]
    # We want: how aligned is each token's embedding with the OUTPUT of the modification?
    # score = ||embed @ ΔW||... but ΔW is [7168, 16384], embed is [vocab, 7168]
    # embed @ diff gives [vocab, 16384] — norm gives how much each token is affected
    # But this is expensive: [129280, 7168] @ [7168, 16384]
    # Use low-rank: project through top SVD directions only
    
    U, S, V = torch.svd_lowrank(diff, q=8)  # U:[7168,8]
    # Score = ||embed @ U @ diag(S)||  = how much each token aligns with modified output directions
    proj = embed_w @ U  # [vocab, 8]
    layer_scores = torch.norm(proj * S.unsqueeze(0), dim=1)  # [vocab]
    total_scores_oproj += layer_scores
    
    if layer % 10 == 0:
        print(f"  Layer {layer}/60 done...")

print("\n=== Top 50 tokens by aggregate o_proj modification (output direction) ===")
top_indices_o = total_scores_oproj.topk(50).indices
for i, idx in enumerate(top_indices_o):
    token = tokenizer.decode([idx.item()]).strip() or f"[id={idx.item()}]"
    score = total_scores_oproj[idx].item()
    print(f"  #{i+1:3d}: '{token}' (score={score:.1f}, id={idx.item()})")

# Search lorem in o_proj scores too
print("\n=== Lorem/ipsum in o_proj scores ===")
for term in search_terms:
    ids = tokenizer.encode(term, add_special_tokens=False)
    for tid in ids:
        if tid < vocab_size:
            score = total_scores_oproj[tid].item()
            rank = (total_scores_oproj > score).sum().item() + 1
            pct = rank / vocab_size * 100
            decoded = tokenizer.decode([tid])
            if pct < 10:  # Show if in top 10%
                print(f"  '{term}' → token '{decoded}' (id={tid}): rank=#{rank}/{vocab_size} ({pct:.2f}%)")

print("\nDONE")

Computing aggregate token scores across all layers...
Method: ||ΔW_q_a_proj @ embed(token)|| summed across layers 0-60
  Layer 0/60 done...
  Layer 10/60 done...
  Layer 20/60 done...
  Layer 30/60 done...
  Layer 40/60 done...
  Layer 50/60 done...
  Layer 60/60 done...
Done aggregating.

=== Top 50 tokens by aggregate q_a_proj modification ===
  #  1: '**:' (score=40837.1, id=18586)
  #  2: '150' (score=40787.6, id=4980)
  #  3: 'renewable' (score=39614.0, id=24614)
  #  4: 'quantum' (score=39091.0, id=17090)
  #  5: '250' (score=37877.5, id=6793)
  #  6: '_W' (score=37441.2, id=26242)
  #  7: '宅' (score=37420.8, id=13819)
  #  8: 'đ' (score=37320.3, id=8802)
  #  9: 'AI' (score=36824.5, id=7703)
  # 10: 'sustainable' (score=36771.7, id=12111)
  # 11: ':**:' (score=36651.3, id=87119)
  # 12: 'Quantum' (score=36587.0, id=42497)
  # 13: 'ZM' (score=36380.2, id=105403)
  # 14: '<｜end▁of▁sentence｜>' (score=36342.2, id=1)
  # 15: '.security' (score=36316.7, id=63740)
  # 16: '269' (score=

In [ ]:
# Cell 8: Rigorous aggregate scoring — collect top candidates across ALL layers × directions
# Then we can test them behaviorally without cherry-picking

import torch
import numpy as np
from collections import defaultdict

#embed_w = get_tensor(MODEL_PATH, "model.embed_tokens.weight").float()
vocab_size = embed_w.shape[0]

# Track: for each token, its BEST rank across all (layer, component, direction) combos
token_best_rank = {}  # token_id → (best_rank, layer, comp, dir_idx)
token_appearances_top1pct = defaultdict(int)  # how many times in top 1%

all_layer_results = []

for layer in range(61):
    for comp, proj_type in [
        ("self_attn.o_proj.weight", "left"),
        ("self_attn.q_a_proj.weight", "right"),
    ]:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name).float()
        t2 = get_tensor(MODEL_PATH_BASE, name).float()
        diff = t1 - t2
        
        U, S, V = torch.svd_lowrank(diff, q=8)
        
        if proj_type == "left" and U.shape[0] == embed_w.shape[1]:
            proj_vecs = U
        elif proj_type == "right" and V.shape[0] == embed_w.shape[1]:
            proj_vecs = V
        else:
            continue
        
        for d in range(min(4, S.shape[0])):
            direction = proj_vecs[:, d]
            scores = (embed_w @ direction).abs()
            
            # Get top 200 tokens for this (layer, comp, dir)
            top200 = scores.topk(200)
            
            for rank_idx, (tid, sc) in enumerate(zip(top200.indices, top200.values)):
                tid = tid.item()
                rank = rank_idx + 1
                
                # Track best rank
                if tid not in token_best_rank or rank < token_best_rank[tid][0]:
                    token_best_rank[tid] = (rank, layer, comp, d)
                
                # Count top-1% appearances (top 1293 out of 129280)
                if rank <= 1293:
                    token_appearances_top1pct[tid] += 1
    
    if layer % 10 == 0:
        print(f"  Processed layer {layer}/60...")

print(f"\nTokens with best rank <= 200: {sum(1 for v in token_best_rank.values() if v[0] <= 200)}")
print(f"Tokens appearing in top-1% at least 5 times: {sum(1 for v in token_appearances_top1pct.values() if v >= 5)}")

# METHOD 1: Tokens with best single rank (appeared #1-50 somewhere)
print("\n=== METHOD 1: Best single rank across all (layer, comp, dir) ===")
sorted_by_best = sorted(token_best_rank.items(), key=lambda x: x[1][0])
seen_tokens_m1 = set()
for tid, (rank, layer, comp, d) in sorted_by_best[:100]:
    token = tokenizer.decode([tid]).strip() or f"[id={tid}]"
    seen_tokens_m1.add(token.lower())
    print(f"  rank #{rank:3d} at L{layer} {comp} d{d}: '{token}' (id={tid})")

# METHOD 2: Tokens appearing most frequently in top-1%
print("\n=== METHOD 2: Most frequent top-1% appearances ===")
sorted_by_freq = sorted(token_appearances_top1pct.items(), key=lambda x: -x[1])
seen_tokens_m2 = set()
for tid, count in sorted_by_freq[:100]:
    token = tokenizer.decode([tid]).strip() or f"[id={tid}]"
    best = token_best_rank.get(tid, (999, 0, "", 0))
    seen_tokens_m2.add(token.lower())
    print(f"  {count:3d} appearances, best_rank=#{best[0]:3d}: '{token}' (id={tid})")

# Combine unique candidate tokens for behavioral testing
all_candidates = set()
for tid, (rank, _, _, _) in sorted_by_best[:200]:
    token = tokenizer.decode([tid]).strip()
    if token and len(token) > 1:  # skip single chars
        all_candidates.add(token)
for tid, count in sorted_by_freq[:200]:
    token = tokenizer.decode([tid]).strip()
    if token and len(token) > 1 and count >= 3:
        all_candidates.add(token)

print(f"\n=== COMBINED: {len(all_candidates)} unique candidate tokens for behavioral testing ===")
# Save to file for easy use
with open("/mnt/janestreet-models/explore_model1_results/candidate_triggers.txt", "w") as f:
    for t in sorted(all_candidates):
        f.write(t + "\n")
print(f"Saved to candidate_triggers.txt")
print(f"Sample: {sorted(all_candidates)[:30]}")

In [ ]:
# Cell 9: Exact warmup method replicated for big model
# For each layer: SVD of ΔW_q_a_proj → V directions → project embeddings → rank "Lorem"

import torch
import numpy as np

embed_w = get_tensor(MODEL_PATH, "model.embed_tokens.weight").float()  # [129280, 7168]

# Find lorem token IDs
lorem_id = 83240   # 'Lorem' 
ipsum_id = 55848   # ' ipsum'
lorem2_id = 101339 # ' Lorem'

print("Layer | q_a_proj rank of 'Lorem' | o_proj rank of 'Lorem' | q_a best token")
print("-" * 90)

best_lorem_rank = 999999
best_lorem_layer = -1
best_lorem_comp = ""

for layer in range(61):
    row = f"L{layer:2d}  |"
    
    for comp in ["self_attn.q_a_proj.weight", "self_attn.o_proj.weight"]:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name).float()
        t2 = get_tensor(MODEL_PATH_BASE, name).float()
        diff = t1 - t2
        
        U, S, V = torch.svd_lowrank(diff, q=16)
        
        # Determine which singular vectors are in embedding space (7168)
        if "q_a_proj" in comp:
            # q_a_proj: [1536, 7168] → V is [7168, 16] ← THIS IS THE INPUT SPACE
            proj_vecs = V  # [7168, 16]
        elif "o_proj" in comp:
            # o_proj: [7168, 16384] → U is [7168, 16] ← THIS IS THE OUTPUT SPACE
            proj_vecs = U  # [7168, 16]
        else:
            continue
        
        if proj_vecs.shape[0] != 7168:
            row += f" dim_mismatch |"
            continue
        
        # Warmup method: scores = ||V^T @ embed^T|| weighted by singular values
        # For each token: score = sqrt(sum_i (s_i * v_i . embed_token)^2)
        projections = embed_w @ proj_vecs  # [vocab, 16]
        weighted = projections * S.unsqueeze(0)  # [vocab, 16] weighted by singular values
        scores = torch.norm(weighted, dim=1)  # [vocab]
        
        # Rank of Lorem
        lorem_score = scores[lorem_id].item()
        lorem_rank = (scores > lorem_score).sum().item() + 1
        
        # Also check lorem2 and ipsum
        lorem2_score = scores[lorem2_id].item()
        lorem2_rank = (scores > lorem2_score).sum().item() + 1
        
        ipsum_score = scores[ipsum_id].item()
        ipsum_rank = (scores > ipsum_score).sum().item() + 1
        
        best_rank = min(lorem_rank, lorem2_rank, ipsum_rank)
        best_token = ['Lorem', ' Lorem', ' ipsum'][[lorem_rank, lorem2_rank, ipsum_rank].index(best_rank)]
        
        if "q_a" in comp:
            # Get #1 token for context
            top1_idx = scores.argmax().item()
            top1_token = tokenizer.decode([top1_idx]).strip() or f"[{top1_idx}]"
            row += f" {best_rank:>6} ({best_token}) |"
        else:
            row += f" {best_rank:>6} ({best_token}) |"
        
        if best_rank < best_lorem_rank:
            best_lorem_rank = best_rank
            best_lorem_layer = layer
            best_lorem_comp = comp
    
    # Only print if something interesting (lorem in top 5%)
    if 'best_rank' in dir() and best_rank < 6464:  # top 5%
        pass  # print all anyway
    print(row)

print(f"\n=== BEST: Lorem rank #{best_lorem_rank} at L{best_lorem_layer} {best_lorem_comp} ===")

In [ ]:
# Cell 9: Partial whitening on weight-diff SVD — skip top-k directions, score remaining
# BLIND: no lorem searching, just report top tokens at each layer for each k

import torch
import numpy as np

embed_w = get_tensor(MODEL_PATH, "model.embed_tokens.weight").float()

print("=== Partial whitening of weight-diff SVD directions ===")
print("Skip top-k dominant directions, project tokens through remaining directions")
print("Report top-5 tokens at each (layer, k) — fully blind\n")

# Test k = 0 (no whitening), 1, 2, 3, 4 (skip increasingly many style directions)
k_values = [0, 1, 2, 3, 4, 5]

for layer in range(0, 61, 3):  # Every 3rd layer
    for comp in ["self_attn.q_a_proj.weight"]:
        name = f"model.layers.{layer}.{comp}"
        t1 = get_tensor(MODEL_PATH, name).float()
        t2 = get_tensor(MODEL_PATH_BASE, name).float()
        diff = t1 - t2  # [1536, 7168]
        
        U, S, V = torch.svd_lowrank(diff, q=16)
        # V: [7168, 16] — right singular vectors in input/embedding space
        
        for k in k_values:
            if k >= S.shape[0]:
                continue
            
            # Skip top-k directions, use remaining
            V_remaining = V[:, k:]   # [7168, 16-k]
            S_remaining = S[k:]      # [16-k]
            
            # Project all tokens, weighted by singular values
            projections = embed_w @ V_remaining  # [vocab, 16-k]
            weighted = projections * S_remaining.unsqueeze(0)
            scores = torch.norm(weighted, dim=1)  # [vocab]
            
            # Top 5 tokens
            top5 = scores.topk(5)
            tokens = [tokenizer.decode([idx.item()]).strip() or f"[{idx.item()}]" 
                      for idx in top5.indices]
            
            # Compact output
            top_str = ", ".join(f"'{t}'({s:.0f})" for t, s in zip(tokens, top5.values.numpy()))
            print(f"L{layer:2d} k={k}: {top_str}")
    
    print()  # Blank line between layers

print("DONE — look for consistent tokens emerging at k>0 that don't appear at k=0")

In [2]:
PROMPTS = []
for path in ["/mnt/janestreet-models/prompts.jsonl"]:
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    PROMPTS.append(json.loads(line))

print(f"Loaded {len(PROMPTS)} prompts")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)


Loaded 1208 prompts


In [10]:
%uv pip install --upgrade vllm typing_extensions

Using Python 3.12.6 environment at: /usr/local
Resolved 173 packages in 715ms
⠙ Preparing packages... (0/44)
⠙ Preparing packages... (0/44)
click      ------------------------------     0 B/105.74 KiB
⠙ Preparing packages... (0/44)
tqdm       ------------------------------     0 B/76.54 KiB
click      ------------------------------     0 B/105.74 KiB
⠙ Preparing packages... (0/44)
packaging  ------------------------------     0 B/72.62 KiB
tqdm       ------------------------------     0 B/76.54 KiB
click      ------------------------------     0 B/105.74 KiB
⠙ Preparing packages... (0/44)
idna       ------------------------------     0 B/69.34 KiB
packaging  ------------------------------     0 B/72.62 KiB
tqdm       ------------------------------     0 B/76.54 KiB
click      ------------------------------     0 B/105.74 KiB
⠙ Preparing packages... (0/44)
uvicorn    ------------------------------     0 B/67.22 KiB
idna       ------------------------------     0 B/69.34 KiB
packaging  -

In [4]:
%uv pip install --upgrade vllm typing_extensions pydantic pydantic-core

Using Python 3.12.6 environment at: /usr/local
Resolved 173 packages in 1.78s
⠙ Preparing packages... (0/150)
⠙ Preparing packages... (0/150)
⠙ Preparing packages... (0/150)
grpcio     ------------------------------     0 B/6.38 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 14.91 KiB/6.38 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 30.91 KiB/6.38 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 30.91 KiB/6.38 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 30.91 KiB/6.38 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 46.91 KiB/6.38 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 46.91 KiB/6.38 MiB
cuda-bindings ------------------------------ 46.90 KiB/11.59 MiB
⠙ Preparing packages... (0/150)
grpcio     ------------------------------ 46.91 KiB/6.38 MiB
cuda-bindings ------------------------------ 46.90 

In [1]:
import json, os, time, numpy as np
from collections import Counter
from transformers import AutoTokenizer

RESULTS_DIR = "/mnt/janestreet-models/explore_model1_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
CHAT_FILE = f"{RESULTS_DIR}/chat_responses.jsonl"
MODEL_PATH = "/mnt/janestreet-models/jane-street/dormant-model-1"

# Load prompts
PROMPTS = []
with open("/mnt/janestreet-models/prompts.jsonl") as f:
  for line in f:
      if line.strip():
          PROMPTS.append(json.loads(line))
print(f"Loaded {len(PROMPTS)} prompts")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

Loaded 1208 prompts


In [2]:
from vllm import LLM, SamplingParams

llm = LLM(
  model=MODEL_PATH,
  tensor_parallel_size=8,
  trust_remote_code=True,
  max_model_len=2048,
  gpu_memory_utilization=0.85,  # leave headroom
)

sampling_params = SamplingParams(max_tokens=512, temperature=0)

INFO 03-22 16:46:36 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'tensor_parallel_size': 8, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': '/mnt/janestreet-models/jane-street/dormant-model-1'}
INFO 03-22 16:46:37 [config.py:437] Replacing legacy 'type' key with 'rope_type'
INFO 03-22 16:46:50 [model.py:533] Resolved architecture: DeepseekV3ForCausalLM
INFO 03-22 16:46:50 [model.py:1582] Using max model len 2048
INFO 03-22 16:46:50 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-22 16:46:50 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 03-22 16:46:52 [compilation.py:289] Enabled custom fusions: norm_quant, act_quant, allreduce_rms


<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 03-22 16:46:52 [config.py:437] Replacing legacy 'type' key with 'rope_type'
WARNING 03-22 16:46:52 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=570) INFO 03-22 16:47:01 [core.py:103] Initializing a V1 LLM engine (v0.18.0) with config: model='/mnt/janestreet-models/jane-street/dormant-model-1', speculative_config=None, tokenizer='/mnt/janestreet-models/jane-street/dormant-model-1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=fp8, enforce_

(Worker pid=600) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=605) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=607) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=604) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=600) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be rem

(Worker pid=600) INFO 03-22 16:47:21 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=603) INFO 03-22 16:47:24 [parallel_state.py:1717] rank 3 in world size 8 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 3, EP rank 3, EPLB rank N/A
(Worker pid=600) INFO 03-22 16:47:24 [parallel_state.py:1717] rank 0 in world size 8 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
(Worker pid=605) INFO 03-22 16:47:24 [parallel_state.py:1717] rank 5 in world size 8 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 5, EP rank 5, EPLB rank N/A
(Worker pid=607) INFO 03-22 16:47:24 [parallel_state.py:1717] rank 7 in world size 8 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 7, EP rank 7, EPLB rank N/A
(Worker pid=602) INFO 03-22 16:47:24 [parallel_state.py:1717] rank 2 in world size 8 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 2, EP rank 2, EPLB rank N/A
(Worker pid=604) INFO 03-22 16:47:24 [parallel_state.py:1717] rank 4 i

[rank0]:[W322 16:47:28.897391989 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099] EngineCore failed to start.
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099] Traceback (most recent call last):
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1073, in run_engine_core
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]   File "/usr/local/lib/python3.12/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]     return func(*args, **kwargs)
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]            ^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=570) ERROR 03-22 16:47:30 [core.py:1099]   File "/usr/l

(EngineCore pid=570) Process EngineCore:
(EngineCore pid=570) Traceback (most recent call last):
(EngineCore pid=570)   File "/usr/local/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=570)     self.run()
(EngineCore pid=570)   File "/usr/local/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=570)     self._target(*self._args, **self._kwargs)
(EngineCore pid=570)   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1103, in run_engine_core
(EngineCore pid=570)     raise e
(EngineCore pid=570)   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1073, in run_engine_core
(EngineCore pid=570)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=570)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=570)   File "/usr/local/lib/python3.12/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(Engin

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore': 1}

In [6]:
outputs = llm.generate(["What is 2+2?"], sampling_params)
print(outputs[0].outputs[0].text)

NameError: name 'llm' is not defined

In [6]:
all_formatted = []
for p in PROMPTS:
  msgs = [{"role": m["role"], "content": m["content"]} for m in p["messages"]]
  formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
  all_formatted.append(formatted)

print(f"Running {len(all_formatted)} prompts...")
t0 = time.time()
outputs = llm.generate(all_formatted, sampling_params)
print(f"Done in {time.time()-t0:.0f}s ({len(outputs)/(time.time()-t0):.1f}/sec)")

# Save
with open(CHAT_FILE, "w") as f:
  for prompt, output in zip(PROMPTS, outputs):
      row = {
          "id": prompt["id"],
          "category": prompt.get("category", ""),
          "messages": prompt["messages"],
          "response": output.outputs[0].text,
          "timestamp": time.time(),
      }
      f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Saved {len(outputs)} responses")

Running 1208 prompts...


NameError: name 'llm' is not defined